In [2]:
import pandas as pd
from pathlib import Path

files = sorted(Path("../data/raw").glob("*.csv"))
print(len(files), "fichiers")

dfs = []
for f in files:
    d = pd.read_csv(f, low_memory=False)
    d.columns = d.columns.str.strip()
    d["source_file"] = f.name
    dfs.append(d)
    print(f.name, d.shape)

df = pd.concat(dfs, ignore_index=True)
print(df.shape)
print(df["Label"].value_counts())

8 fichiers
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (225745, 80)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (286467, 80)
Friday-WorkingHours-Morning.pcap_ISCX.csv (191033, 80)
Monday-WorkingHours.pcap_ISCX.csv (529918, 80)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv (288602, 80)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (170366, 80)
Tuesday-WorkingHours.pcap_ISCX.csv (445909, 80)
Wednesday-workingHours.pcap_ISCX.csv (692703, 80)
(2830743, 80)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web At

In [3]:
import numpy as np

# nettoyage
df["Label"] = df["Label"].str.replace("\ufffd", "-", regex=False).str.replace("Web Attack - ", "Web Attack ", regex=False).str.strip()

# absurde x manquante
num = df.select_dtypes(include=[np.number])
print("Inf :", np.isinf(num).sum().sum())
print("NaN :", df.isna().sum().sum())
print(df.isna().sum()[df.isna().sum() > 0])

cols_inf = np.isinf(num).sum()
print(cols_inf[cols_inf > 0])

# les doublons
feat_cols = [c for c in df.columns if c != "source_file"]
dups = df.duplicated(subset=feat_cols).sum()
print("Doublons :", dups, f"({dups/len(df):.1%})")

# les colonnes constantes
const = [c for c in num.columns if num[c].nunique() <= 1]
print("Colonnes constantes :", const)

# calcule du desiquilibre
print((df["Label"].value_counts(normalize=True) * 100).round(3))

Inf : 4376
NaN : 1358
Flow Bytes/s    1358
dtype: int64
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64
Doublons : 308381 (10.9%)
Colonnes constantes : ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Label
BENIGN                      80.300
DoS Hulk                     8.163
PortScan                     5.614
DDoS                         4.523
DoS GoldenEye                0.364
FTP-Patator                  0.280
SSH-Patator                  0.208
DoS slowloris                0.205
DoS Slowhttptest             0.194
Bot                          0.069
Web Attack Brute Force       0.053
Web Attack XSS               0.023
Infiltration                 0.001
Web Attack Sql Injection     0.001
Heartbleed                   0.000
Name: proportion, dtype: float64
